# Modelling — Krakow PM2.5 Spatial Regression

**Input:** `data/output/krakow_final_dataset.csv` — 13,597 daily rows, 8 stations, 79 columns

**Outputs:**
- `data/processed/krakow-pm25-{train,val,test}.csv` — station-group split
- `models/baseline.joblib` — saved sklearn Pipeline
- `docs/model-cards/krakow-pm25-spatial-rf-v1.md` — the model certificate

**Split:** leave-one-station-out. Tests generalization to new locations — the only split matching deployment (32,700 grid cells with no monitors).

**Features:** land use at 500m + 1km + temporal. NOT lag features (pm25_lag1 etc.) — those require knowing previous PM2.5 at the target location, unavailable for unmeasured grid cells.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import joblib
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

CLEAN_PATH = Path("../data/output/krakow_final_dataset.csv")
SPLIT_DIR  = Path("../data/processed/")
MODEL_PATH = Path("../models/baseline.joblib")
SLUG       = "krakow-pm25"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

TARGET = "pm25"

# Land use + temporal only — no lag features (not available at unmeasured locations)
FEATURE_COLS = [
    "luse_r005_urban_pct",
    "luse_r005_green_pct",
    "luse_r005_urban_industrial_pct",
    "luse_r005_seal_density",
    "luse_r010_urban_pct",
    "luse_r010_green_pct",
    "luse_r010_urban_industrial_pct",
    "luse_r010_seal_density",
    "month",
    "month_sin",
    "month_cos",
    "season_winter",
    "is_weekend",
]

TEST_STATIONS = frozenset({"zloty_rog", "nowa_huta"})
VAL_STATIONS  = frozenset({"kurdwanow"})

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
print("Config OK")

In [ ]:
df = pd.read_csv(CLEAN_PATH)
print(f"Shape: {df.shape}")
print(f"Stations ({df['station_id'].nunique()}): {sorted(df['station_id'].unique())}")
print(f"Years: {sorted(df['year'].unique())}")
print(f"Target (pm25) — min: {df['pm25'].min():.1f}  max: {df['pm25'].max():.1f}  mean: {df['pm25'].mean():.1f} µg/m³")
df.head(3)

In [ ]:
assert df['station_id'].nunique() == 8, f"Expected 8 stations, got {df['station_id'].nunique()}"
assert (df['pm25'] >= 0).all(), "Negative pm25 — cleaning contract broken"

missing_features = [c for c in FEATURE_COLS if c not in df.columns]
if missing_features:
    print(f"WARNING: Missing feature columns: {missing_features}")
else:
    print("All feature columns present")

nan_pct = df[FEATURE_COLS].isna().mean().sort_values(ascending=False)
print("NaN % per feature:")
print(nan_pct[nan_pct > 0].to_string() or "  (no NaNs)")

## Task 1: Split

**Decision: leave-one-station-out (LOSO) group split.**

- Random/temporal split: all 8 stations in train and test — model evaluated on stations it trained on. Tests nothing about spatial generalization.
- **Deployment reality:** model predicts at 32,700 grid cells with no monitors. LOSO tests the right question: can the model predict at a location it has never seen?
- Lag features (pm25_lag1/lag3/lag7/roll7d) excluded from FEATURE_COLS: at deployment time there is no previous PM2.5 at an unmeasured location.

| Set | Stations | ~Rows |
|---|---|---|
| train | 5 stations | ~8,500 |
| val | kurdwanow | ~1,700 |
| test | zloty_rog + nowa_huta | ~3,400 — SACRED |

In [ ]:
train_stations = set(df['station_id'].unique()) - TEST_STATIONS - VAL_STATIONS

train = df[df['station_id'].isin(train_stations)].copy().reset_index(drop=True)
val   = df[df['station_id'].isin(VAL_STATIONS)].copy().reset_index(drop=True)
test  = df[df['station_id'].isin(TEST_STATIONS)].copy().reset_index(drop=True)

assert not (set(train['station_id']) & set(val['station_id'])),  "leak train<->val"
assert not (set(train['station_id']) & set(test['station_id'])), "leak train<->test"
assert not (set(val['station_id'])   & set(test['station_id'])), "leak val<->test"

print(f"train: {len(train_stations)} stations · {len(train):,} rows — {sorted(train_stations)}")
print(f"val:   {sorted(VAL_STATIONS)} · {len(val):,} rows")
print(f"test:  {sorted(TEST_STATIONS)} · {len(test):,} rows  <- SACRED")
print("Leakage assertions passed.")

In [ ]:
for name, part in [("train", train), ("val", val), ("test", test)]:
    out = SPLIT_DIR / f"{SLUG}-{name}.csv"
    part.to_csv(out, index=False)
    print(f"Wrote {out}  ({len(part):,} rows · {out.stat().st_size / 1024:.0f} KB)")
print("\nTest CSV is now sacred. Do NOT open it until the model is locked.")

## Task 2: Baselines — the floor

1. **Dumb mean:** predict training-set mean pm25 for every row. Unconditional floor.
2. **Seasonal mean:** predict month-specific mean from training data. Captures Krakow's heating-season swing (winter ~58 µg/m³ vs summer ~12 µg/m³). **If RF can't beat this, it isn't learning land use — it's re-discovering that January is polluted.**

In [ ]:
X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET]  # defined but SACRED

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape} (sacred)")
print(f"y_train: mean={y_train.mean():.1f}  std={y_train.std():.1f}  min={y_train.min():.0f}  max={y_train.max():.0f} µg/m³")

In [ ]:
dumb = DummyRegressor(strategy="mean")
dumb.fit(X_train, y_train)
dumb_val_mae = mean_absolute_error(y_val, dumb.predict(X_val))
print(f"Dumb-mean (predicts {dumb.constant_[0]:.1f} µg/m³ everywhere):")
print(f"  train MAE = {mean_absolute_error(y_train, dumb.predict(X_train)):.2f}")
print(f"  val   MAE = {dumb_val_mae:.2f}")

In [ ]:
month_means = train.groupby("month")[TARGET].mean()
print("Month means (µg/m³):"); print(month_means.round(1).to_string())

def predict_seasonal(month_means, df):
    return df["month"].map(month_means).fillna(month_means.mean())

seas_train_preds = predict_seasonal(month_means, train)
seas_val_preds   = predict_seasonal(month_means, val)
seas_val_mae     = mean_absolute_error(y_val, seas_val_preds)

print(f"\nSeasonal baseline:")
print(f"  train MAE = {mean_absolute_error(y_train, seas_train_preds):.2f}  R² = {r2_score(y_train, seas_train_preds):.3f}")
print(f"  val   MAE = {seas_val_mae:.2f}  R² = {r2_score(y_val, seas_val_preds):.3f}")
print(f"  Beats dumb by {dumb_val_mae - seas_val_mae:.1f} µg/m³. RF must beat seasonal.")

## Task 3: One defensible model

1. **Problem shape:** regression from 13 land-use + temporal features.
2. **Data size:** ~8,500 training rows — sufficient for RF; too small for deep learning.
3. **Explainability:** yes — planning analysts (Session 7) ask 'why is this location high?'

**Chosen:** `RandomForestRegressor(n_estimators=300, min_samples_leaf=5)`

**Why not linear regression:** land-use × season interaction is nonlinear. Expected R² < 0.35.

**Why not Gradient Boosting:** reserved for Session 5. Session 4 establishes the defensible baseline.

In [ ]:
pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
    ("model",  RandomForestRegressor(
        n_estimators=300, min_samples_leaf=5,
        random_state=RANDOM_SEED, n_jobs=-1,
    )),
])
pipeline.fit(X_train, y_train)
print("Pipeline fitted.")

In [ ]:
model_train_mae = mean_absolute_error(y_train, pipeline.predict(X_train))
model_val_mae   = mean_absolute_error(y_val,   pipeline.predict(X_val))
model_train_r2  = r2_score(y_train, pipeline.predict(X_train))
model_val_r2    = r2_score(y_val,   pipeline.predict(X_val))

print(f"RF model — train MAE: {model_train_mae:.2f}  R²: {model_train_r2:.3f}")
print(f"RF model — val   MAE: {model_val_mae:.2f}  R²: {model_val_r2:.3f}")
print(f"Beats seasonal by {seas_val_mae - model_val_mae:.1f} µg/m³ on val.")

if model_val_r2 >= 0.40:
    print(f"Brief criterion MET: val R² = {model_val_r2:.3f} >= 0.40")
else:
    print(f"Brief criterion NOT MET: val R² = {model_val_r2:.3f} < 0.40")

In [ ]:
report = pd.DataFrame([
    {"split": "train", "model": "dumb-mean",  "MAE": round(mean_absolute_error(y_train, dumb.predict(X_train)), 2), "R2": round(r2_score(y_train, dumb.predict(X_train)), 3)},
    {"split": "train", "model": "seasonal",   "MAE": round(mean_absolute_error(y_train, seas_train_preds), 2),      "R2": round(r2_score(y_train, seas_train_preds), 3)},
    {"split": "train", "model": "RF (ours)",  "MAE": round(model_train_mae, 2), "R2": round(model_train_r2, 3)},
    {"split": "val",   "model": "dumb-mean",  "MAE": round(dumb_val_mae, 2),   "R2": round(r2_score(y_val, dumb.predict(X_val)), 3)},
    {"split": "val",   "model": "seasonal",   "MAE": round(seas_val_mae, 2),   "R2": round(r2_score(y_val, seas_val_preds), 3)},
    {"split": "val",   "model": "RF (ours)",  "MAE": round(model_val_mae, 2),  "R2": round(model_val_r2, 3)},
])
print(report.to_string(index=False))
print("\nTest split NOT shown — touched once after model locked (cell c19).")

## Task 4: Assess

### 4a: LOSO cross-validation (6 train+val stations)

In [ ]:
train_val     = pd.concat([train, val], ignore_index=True)
loso_stations = train_val['station_id'].unique()

fold_rows = []
for held_out in loso_stations:
    ftr = train_val[train_val['station_id'] != held_out]
    fvl = train_val[train_val['station_id'] == held_out]
    p = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
        ("model",  RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                          random_state=RANDOM_SEED, n_jobs=-1)),
    ])
    p.fit(ftr[FEATURE_COLS], ftr[TARGET])
    preds = p.predict(fvl[FEATURE_COLS])
    fold_rows.append({
        "held_out": held_out, "n_val": len(fvl),
        "MAE": round(mean_absolute_error(fvl[TARGET], preds), 2),
        "R2":  round(r2_score(fvl[TARGET], preds), 3),
    })

cv_results = pd.DataFrame(fold_rows).sort_values("MAE", ascending=False)
print(cv_results.to_string(index=False))
print(f"\nLOSO-CV MAE: {cv_results['MAE'].mean():.2f} +/- {cv_results['MAE'].std():.2f} µg/m³")
print(f"LOSO-CV R²:  {cv_results['R2'].mean():.3f} +/- {cv_results['R2'].std():.3f}")
print("\nWorst stations -> Session 6 failure gallery candidates:")
print(cv_results.head(2)[['held_out','MAE','R2']].to_string(index=False))

In [ ]:
forest     = pipeline.named_steps["model"]
Xt_val     = pipeline[:-1].transform(X_val)
tree_preds = np.stack([t.predict(Xt_val) for t in forest.estimators_])
y_lo = np.percentile(tree_preds, 5,  axis=0)
y_hi = np.percentile(tree_preds, 95, axis=0)
y_pt = tree_preds.mean(axis=0)

coverage = np.mean((y_val.values >= y_lo) & (y_val.values <= y_hi))
print(f"90% interval — coverage: {coverage:.1%}  (target >= 85%)  width: {(y_hi - y_lo).mean():.1f} µg/m³")
if coverage < 0.85:
    print("WARNING: Under-coverage — intervals overconfident.")

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
idx = np.argsort(val['year'].values * 366 + val['day_of_year'].values)
ax.plot(range(len(idx)), y_val.values[idx], 'k-', lw=0.7, label='Observed')
ax.plot(range(len(idx)), y_pt[idx], 'b--', lw=0.7, label='RF predicted')
ax.fill_between(range(len(idx)), y_lo[idx], y_hi[idx], alpha=0.2, color='blue', label='90% PI')
ax.set_xlabel('Day (time-ordered)'); ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('Val station (Kurdwanow) — observed vs predicted')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
val_pred = pipeline.predict(X_val)
val_aug  = val.assign(pred=val_pred, abs_err=np.abs(val_pred - y_val))
val_aug['season'] = val_aug['month'].map({12:'W',1:'W',2:'W',3:'Sp',4:'Sp',5:'Sp',
                                           6:'Su',7:'Su',8:'Su',9:'A',10:'A',11:'A'})
print("Per-season MAE on val (Kurdwanow):")
print(val_aug.groupby('season')['abs_err'].agg(['mean','count']).rename(columns={'mean':'MAE','count':'n'}).round(2).to_string())

importances = pd.Series(pipeline.named_steps['model'].feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print("\nFeature importances:")
print(importances.round(3).to_string())
print("\nPer-station LOSO-CV:")
print(cv_results.to_string(index=False))

> If temporal features dominate importance, the model is learning seasonality more than land use. At least one of `luse_r005_*` or `luse_r010_*` should appear in the top 5 importances for the model to be credible as a land-use regression.

## Task 5: Lock the model — touch test once

In [ ]:
joblib.dump(pipeline, MODEL_PATH)
loaded = joblib.load(MODEL_PATH)
assert np.allclose(loaded.predict(X_val), pipeline.predict(X_val)), "round-trip failed"
print(f"Saved {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1024:.1f} KB) — round-trip OK")
print("Model is LOCKED. Run cell c19 only once.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FINAL TEST SCORE — run ONCE after model locked and committed.
# ─────────────────────────────────────────────────────────────────────────────
test_pred     = pipeline.predict(X_test)
test_mae      = mean_absolute_error(y_test, test_pred)
test_r2       = r2_score(y_test, test_pred)
dumb_test_mae = mean_absolute_error(y_test, dumb.predict(X_test))
seas_test_mae = mean_absolute_error(y_test, predict_seasonal(month_means, test))

Xt_test    = pipeline[:-1].transform(X_test)
tree_test  = np.stack([t.predict(Xt_test) for t in forest.estimators_])
test_cov   = np.mean((y_test.values >= np.percentile(tree_test, 5, axis=0)) &
                     (y_test.values <= np.percentile(tree_test, 95, axis=0)))

print("=" * 50)
print("FINAL TEST RESULTS (zloty_rog + nowa_huta)")
print("=" * 50)
print(f"  dumb-mean MAE:  {dumb_test_mae:.2f} µg/m³")
print(f"  seasonal  MAE:  {seas_test_mae:.2f} µg/m³")
print(f"  RF (ours) MAE:  {test_mae:.2f} µg/m³   R²: {test_r2:.3f}")
print(f"  90% coverage:   {test_cov:.1%}")
print()
print(f"Brief criterion {'MET' if test_r2 >= 0.40 else 'NOT MET'}: test R² = {test_r2:.3f}")
print()
print("NOTE: zloty_rog reads 20-30% low (sheltered bias) — caveat in model card section 3.")
print("=" * 50)

## What gets promoted to `src/`?

`src/split_data.py` — `make_station_splits`, `make_loso_folds`, `write_splits`

`src/baseline_model.py` — `build_pipeline`, `train_baseline_model`, `run_loso_cv`, `compute_metrics_table`, `predict_with_uncertainty`, `save_pipeline`

**Pre-commit ritual:**
1. Restart kernel → Run All — no errors
2. `python src/split_data.py` — three CSVs written, assertions pass
3. `python src/baseline_model.py` — `models/baseline.joblib` written, round-trip OK
4. Metrics in model card §7.1 match cell c19 output above